# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Szalone nożyczki (Teatr Bagatela)
2. Atrament dla leworęcznych. Komedia absurdalna
3. Karty na stół
4. Nasze żony
5. Dizajn i emocje – jak działa na nas projektowanie?
6. Martwe natury
7. Bajki dla niegrzecznych
8. Hexvessel, Aluk Todolo, BaarRa w Zaścianku
9. Eros Ramazzotti: Una storia importante w Tauron Arenie Kraków
10. Wróg ludu

Czas wykonania: 5.08s


In [3]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Szalone nożyczki (Teatr Bagatela)
2. Atrament dla leworęcznych. Komedia absurdalna
3. Karty na stół
4. Nasze żony
5. Dizajn i emocje – jak działa na nas projektowanie?
6. Martwe natury
7. Bajki dla niegrzecznych
8. Hexvessel, Aluk Todolo, BaarRa w Zaścianku
9. Eros Ramazzotti: Una storia importante w Tauron Arenie Kraków
10. Wróg ludu

Czas wykonania (wątki): 1.48s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [5]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [7]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 8 procesach (rdzeniach)...
Zakończono w czasie 1.29s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [9]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

# Twój kod tutaj...
def get_cat_fact():
    return requests.get(CAT_API_URL).json().get("fact")

#sekwencyjnie
def run_sequential():
    print("sekwencyjnie:")
    start = time.time()

    facts = []
    for _ in range(20):
        facts.append(get_cat_fact())

    end = time.time()

    for i, fact in enumerate(facts[:5], 1):
        print(f"{i}. {fact}")

    print(f"\nCzas: {end - start:.2f}s\n")



#wielowątkowo
def run_threaded():
    print("wielowątkowo:")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        facts = list(executor.map(lambda _: get_cat_fact(), range(20)))

    end = time.time()

    for i, fact in enumerate(facts[:5], 1):
        print(f"{i}. {fact}")

    print(f"\nCzas: {end - start:.2f}s\n")



run_sequential()
run_threaded()

sekwencyjnie:
1. A happy cat holds her tail high and steady.
2. Tylenol and chocolate are both poisionous to cats.
3. A cat has the ability to rotate their ears 180 degrees,with the help of 32 muscles that they use to control them.
4. Cats only use their meows to talk to humans, not each other. The only time they meow to communicate with other felines is when they are kittens to signal to their mother.
5. Blue-eyed, pure white cats are frequently deaf.

Czas: 9.46s

wielowątkowo:
1. An adult lion's roar can be heard up to five miles (eight kilometers) away.
2. A cat called Dusty has the known record for the most kittens. She had more than 420 kittens in her lifetime.
3. In multi-cat households, cats of the opposite sex usually get along better.
4. A cat can’t climb head first down a tree because every claw on a cat’s paw points the same way. To get down from a tree, a cat must back down.
5. Cat's urine glows under a black light.

Czas: 1.71s



### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [13]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

# Twój kod tutaj...

q = queue.Queue()
LIMIT = 20
STOP = None #sygnal, ze konsument ma przerwac prace

def producer(): #dodaje liczby do kolejki 
    for i in range(1, LIMIT + 1):
        print(f"producent dodał: {i}")
        q.put(i)
        time.sleep(0.05)

    q.put(STOP)
    q.put(STOP)

def consumer_even(): #bvierze parzyste
    while True:
        item = q.get()

        if item is STOP:
            q.task_done()
            break

        if item % 2 == 0:
            print(f"konsument parzystych pobrał: {item}")
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.01)

def consumer_odd(): #bierze nieparzyste 
    while True:
        item = q.get()

        if item is STOP:
            q.task_done()
            break

        if item % 2 != 0:
            print(f"konsument nieparzystych pobrał: {item}")
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.01)

t_producer = threading.Thread(target=producer)
t_even = threading.Thread(target=consumer_even)
t_odd = threading.Thread(target=consumer_odd)

t_producer.start()
t_even.start()
t_odd.start()

t_producer.join()
q.join()
t_even.join()
t_odd.join()

print("Koniec programu")

producent dodał: 1
konsument nieparzystych pobrał: 1
producent dodał: 2
konsument parzystych pobrał: 2
producent dodał: 3
konsument nieparzystych pobrał: 3
producent dodał: 4
konsument parzystych pobrał: 4
producent dodał: 5
konsument nieparzystych pobrał: 5
producent dodał: 6
konsument parzystych pobrał: 6
producent dodał: 7
konsument nieparzystych pobrał: 7
producent dodał: 8
konsument parzystych pobrał: 8
producent dodał: 9
konsument nieparzystych pobrał: 9
producent dodał: 10
konsument parzystych pobrał: 10
producent dodał: 11
konsument nieparzystych pobrał: 11
producent dodał: 12
konsument parzystych pobrał: 12
producent dodał: 13
konsument nieparzystych pobrał: 13
producent dodał: 14
konsument parzystych pobrał: 14
producent dodał: 15
konsument nieparzystych pobrał: 15
producent dodał: 16
konsument parzystych pobrał: 16
producent dodał: 17
konsument nieparzystych pobrał: 17
producent dodał: 18
konsument parzystych pobrał: 18
producent dodał: 19
konsument nieparzystych pobrał: 19


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [23]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    # Twój kod tutaj...
    start = time.time()

    cores = multiprocessing.cpu_count()
    print(f"praca na {cores} procesach...")

    numbers = list(range(1, 10001))

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)

    end = time.time()

    print(f"Obliczono {len(results)} wyników.")
    print("pierwsze pięć wyników:")
    for i, result in enumerate(results[:5], 1):
        print(f"{i}. {result}")

    print(f"\nCzas wykonania: {end - start:.2f}s")

praca na 8 procesach...
Obliczono 10000 wyników.
pierwsze pięć wyników:
1. 100
2. 2535301200456458802993406410750
3. 773066281098016996554691694648431909053161283000
4. 2142584059011987034055949456454883470029603991710390447068500
5. 9860761315262647567646607066034827870915080438862787559628486633300780

Czas wykonania: 0.48s
